> ## ⚠️ UNEXERCISED — blocked on the Spider 2.0 data gate
>
> **Neither this notebook nor `19_spider2_semantic_layer.ipynb` has ever been run
> end-to-end.** They are committed as the prepared harness, not as a verified
> result.
>
> The blocker is upstream:
> [`scripts/benchmarks/ingest_spider2.py`](../scripts/benchmarks/ingest_spider2.py)
> has `SPIDER2_DATA_URL = ""` (line 50) and fails loud while it is empty — the
> Spider 2.0-lite SQLite subset's download URL and on-disk layout must be
> **confirmed by an operator at a live gate** first. Until then there is no
> `spider2.jsonl` and no completed `vkg-spider2_*` layer, so every cell below has
> nothing to bind to.
>
> **When you open the gate:** set `SPIDER2_DATA_URL`, run the ingest, run nb19 to
> build the layers, *then* run this notebook. Expect the exec metrics to need the
> same kind of gold-SQL triage BIRD and CWD needed (dialect gaps make a fraction of
> gold queries unexecutable on Athena, which lowers the answerable-row count rather
> than the score) — and delete this cell once a full pass is green.



# Spider 2.0 Benchmark — VKG Query-Agent Eval (LLM judges + codeBased exec metrics)

Runs the **Spider 2.0-lite (SQLite subset)** benchmark's natural-language
questions through the deployed **ontology_query** agent (VKG path,
`QueryRuntimeArn`) and scores each answer **two ways, entirely server-side** via
AgentCore Batch Evaluations:

1. **LLM-as-Judge (SESSION)** — `GoalSuccess`, `FinalAnswerFaithfulness`,
   `SqlGrounded` from `agents/shared/eval_judges.py` (VKG family), identical to
   `5_ontology_queryagent_ac_eval.ipynb`.
2. **codeBased exec metrics (TRACE)** — `ExecutionAccuracy` (EX), `RewardVES`
   (R-VES), `SoftF1`, computed by the **BenchmarkEvalHandler** Lambda
   (`BenchmarkEvalHandlerArn`). Each is registered as a `codeBased` evaluator
   pointing at the SAME Lambda; the Lambda branches on the evaluator name. It
   extracts the agent's predicted SQL from the trace spans, reads the **gold
   SQL** from the reference input, executes both on Athena, and returns EX /
   R-VES / Soft-F1.

Spider 2.0 spans **many databases** (one namespace per ingested SQLite DB), so —
like BIRD — each row binds to its own DB's VKG layer `vkg-{row.db_id}` (e.g.
`vkg-spider2_<db>`).

### Prerequisites
- **`19_spider2_semantic_layer.ipynb` built the Spider 2.0 VKG layer(s)** (ids
  `vkg-spider2_<db>`). This notebook filters Spider 2.0 rows to the DBs that have a
  completed layer and invokes the agent with `ontologyId = f"vkg-{row.db_id}"`.
- **`BenchmarkEvalHandler` Lambda deployed** (the AgentCore stack's Part-A
  codeBased evaluator; CFN output `BenchmarkEvalHandlerArn`).
- The Spider 2.0 SQLite subset is ingested (manifest + `spider2.jsonl` under
  `data/eval/benchmarks/spider2/` — see nb19's stretch-tier data-gate note and
  `scripts/benchmarks/ingest_spider2.py`).

### Gold-packing (READ THIS — the dual-scoring contract)
Spider 2.0's gold is **SQL** (transpiled SQLite→Trino by the ingest), not a prose
answer — like BIRD. The codeBased Lambda reads gold from `expectedResponse.text`
as JSON `{"gold_sql", "db_id", "catalog"}`; the nb5 LLM judges read the expected
**NL answer** from `{assertions}` (threaded via `FINAL_ANSWER_ASSERTION_PREFIX`).
These want DIFFERENT things in the same field.

**Resolution:** we pack the gold JSON into each scenario's `expected_response`
(via `parse_multiturn_row` → `build_scenarios`, which puts it on the final
`Turn.expected_response` → the batch API's TRACE `expectedResponse.text`). That
is exactly what the codeBased Lambda reads, so **EX / R-VES / Soft-F1 are
correct** — and they are the primary Spider 2.0 signal. Because Spider 2.0 rows
carry gold SQL, **not** a prose answer, that same packed JSON also becomes the LLM
judge's `FINAL_ANSWER_ASSERTION_PREFIX` reference — which makes
`FinalAnswerFaithfulness` **weak/uninformative for Spider 2.0** (it is comparing
the agent's NL answer to a JSON blob). Accepted trade-off: **the exec metrics are
the primary Spider 2.0 signal**; `GoalSuccess` and `SqlGrounded` (which don't
depend on a gold NL answer) remain meaningful.


In [ ]:
# Prereq (uncomment if not already installed):
# !pip install bedrock-agentcore-starter-toolkit==0.3.9

## 1. Setup & Credentials

In [ ]:
import os
import json
import uuid
import time
import boto3
import pandas as pd
from botocore.config import Config
from datetime import datetime, timezone
from dotenv import load_dotenv
from IPython.display import Markdown, display

# Load .env if present, then set defaults
load_dotenv(dotenv_path='.env', override=False)
os.environ.setdefault('AWS_PROFILE', 'huthmac+demo')
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')

PROJECT_NAME = os.environ.get('PROJECT_NAME', 'semantic-layer')

config = Config(retries={'max_attempts': 10, 'mode': 'standard'}, signature_version='v4')

# Create session with AWS profile
session = boto3.Session(profile_name=os.environ['AWS_PROFILE'])
region = session.region_name or 'us-east-1'

# Verify credentials
sts = session.client('sts', region_name=region, config=config)
identity = sts.get_caller_identity()
print(f"✓ AWS Credentials Verified:")
print(f"  Profile: {os.environ['AWS_PROFILE']}")
print(f"  Account: ...{identity['Account'][-4:]}")
print(f"  User/Role: {identity['Arn'].split('/')[-1]}")
print(f"  Region: {region}")

# Render full cell text in displayed tables — never truncate question/message/SQL
# columns (the default max_colwidth=50 cut ground-truth questions mid-word).
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)


In [ ]:
import re

def _redact_account_ids(obj):
    """Recursively mask AWS account IDs in ARN strings to their last 4 digits (********NNNN)."""
    if isinstance(obj, dict):
        return {k: _redact_account_ids(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_redact_account_ids(v) for v in obj]
    if isinstance(obj, str):
        return re.sub(r'(\d{8})(\d{4})', r'********\2', obj)
    return obj


def _mask_acct(text):
    """Mask AWS account IDs (12-digit runs) to their last 4 digits — e.g. ********2087 —
    in any string (ARNs, bucket names, plain ids). Never emits the full account number."""
    import re as __re
    if not isinstance(text, str):
        return text
    return __re.sub(r'(\d{8})(\d{4})', r'********\2', text)


In [ ]:
# ── OAuth M2M runtime invoker ───────────────────────────────────────────────
# AgentCore runtimes use CUSTOM_JWT (Cognito M2M) inbound auth.
# Invoke the JWT-inbound runtime with a Bearer
# token minted via Cognito client_credentials instead.
import base64
import urllib.parse as _urlparse
import urllib.request as _urlrequest

_BROWSER_UA = (
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
    'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36'
)
_OAUTH_SCOPE = 'semantic-layer-mcp/invoke'


def _get_m2m_creds() -> tuple:
    """Return (token_endpoint, client_id, client_secret) from CFN + Secrets Manager.

    Returns:
        Tuple of (token_endpoint str, client_id str, client_secret str).
    """
    cfn_client = session.client('cloudformation', region_name=region)
    auth_outputs = {}
    for stack_name in (f'{PROJECT_NAME}-auth', f'{PROJECT_NAME}-dev-auth'):
        try:
            stacks = cfn_client.describe_stacks(StackName=stack_name)['Stacks']
            auth_outputs = {o['OutputKey']: o['OutputValue'] for o in stacks[0].get('Outputs', [])}
            break
        except Exception:
            continue
    if not auth_outputs:
        raise RuntimeError(f'Auth stack not found for project {PROJECT_NAME}')
    token_endpoint = auth_outputs['McpHostedUiDomainUrl'] + '/oauth2/token'
    client_id = auth_outputs['M2mClientId']
    lam = session.client('lambda')
    # The deployed function carries the stage infix (e.g. semantic-layer-dev-mcp-tools);
    # try both the bare and the '-dev-' variant, mirroring the auth-stack lookup above.
    cfg = None
    for _fn_name in (f'{PROJECT_NAME}-mcp-tools', f'{PROJECT_NAME}-dev-mcp-tools'):
        try:
            cfg = lam.get_function_configuration(FunctionName=_fn_name)
            break
        except lam.exceptions.ResourceNotFoundException:
            continue
    if cfg is None:
        raise RuntimeError(f'mcp-tools Lambda not found for project {PROJECT_NAME}')
    secret_arn = cfg['Environment']['Variables']['M2M_CLIENT_SECRET_ARN']
    client_secret = session.client('secretsmanager').get_secret_value(
        SecretId=secret_arn,
    )['SecretString']
    return token_endpoint, client_id, client_secret


def _fetch_m2m_token() -> str:
    """Mint a Cognito client_credentials token for agent runtime invocations.

    Returns:
        OAuth access_token string.
    """
    token_endpoint, client_id, client_secret = _get_m2m_creds()
    body = _urlparse.urlencode({
        'grant_type': 'client_credentials',
        'scope': _OAUTH_SCOPE,
    }).encode()
    basic = base64.b64encode(f'{client_id}:{client_secret}'.encode()).decode('ascii')
    req = _urlrequest.Request(
        token_endpoint, data=body, method='POST',
        headers={
            'Content-Type': 'application/x-www-form-urlencoded',
            'Authorization': f'Basic {basic}',
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=30) as resp:
        return json.loads(resp.read())['access_token']


def _invoke_runtime(arn: str, session_id: str, payload: bytes, *, qualifier: str = 'DEFAULT') -> bytes:
    """Invoke an AgentCore Runtime with Cognito Bearer auth.

    Parameters:
        arn: runtime ARN.
        session_id: X-Amzn-Bedrock-AgentCore-Runtime-Session-Id header value.
        payload: JSON-encoded request body.
        qualifier: runtime qualifier (default 'DEFAULT').
    Returns:
        Raw response body bytes.
    """
    token = _fetch_m2m_token()
    encoded_arn = arn.replace(':', '%3A').replace('/', '%2F')
    url = (
        f'https://bedrock-agentcore.{region}.amazonaws.com/'
        f'runtimes/{encoded_arn}/invocations?qualifier={qualifier}'
    )
    req = _urlrequest.Request(
        url, data=payload, method='POST',
        headers={
            'Authorization': f'Bearer {token}',
            'Content-Type': 'application/json',
            'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id,
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=900) as resp:
        return resp.read()


print('✓ OAuth M2M runtime invoker ready')

## 2. Load Spider 2.0 rows + resolve the built VKG layers

In [ ]:
# ── Load Spider 2.0 rows and keep only those with a built VKG layer + gold SQL ─────
import sys
sys.path.insert(0, '..')
from pathlib import Path
from agents.shared.benchmark_adapter import load_rows
from scripts.benchmarks.ingest_common import Manifest

BENCHMARK = 'spider2'
BENCH_DIR = Path(f'../data/eval/benchmarks/{BENCHMARK}')
JSONL = BENCH_DIR / f'{BENCHMARK}.jsonl'

# Manifest maps each db_id (= namespace "spider2_<db>") -> catalog / database /
# tables. We use each row's manifest entry to build the codeBased gold payload
# (the Lambda needs the federated catalog + Athena database to execute gold SQL).
_manifest_path = None
for _cand in (BENCH_DIR / 'manifest.json', BENCH_DIR / 'deploy_manifest.json'):
    if _cand.exists():
        _manifest_path = _cand
        break
if _manifest_path is None:
    raise FileNotFoundError(
        f"No Spider 2.0 manifest under {BENCH_DIR}. Run scripts/benchmarks/"
        f"deploy_datasets.py --benchmark spider2 first."
    )
manifest = Manifest.load(path=_manifest_path)
_ingested_ids = {e.db_id for e in manifest.entries}

all_rows = load_rows(path=JSONL)

# ── Which Spider 2.0 DBs have a COMPLETED VKG layer (built by notebook 19)? ─────────
# nb19 seeds/builds layers with the deterministic id `vkg-<namespace>` where
# namespace == the Spider 2.0 db_id (e.g. "spider2_<db>"). We scan DynamoDB for
# completed vkg-spider2_* layers so this notebook is self-sufficient (it does NOT
# require nb19 to have %store'd anything in this kernel).
METADATA_TABLE = os.environ.get('ONTOLOGY_METADATA_TABLE', f'{PROJECT_NAME}-metadata')
_meta_table = session.resource('dynamodb', region_name=region).Table(METADATA_TABLE)


def _completed_spider2_vkg_layers() -> set:
    """Return the set of db_ids (namespaces) that have a completed vkg-spider2_* layer."""
    done, scan_kw = set(), {
        'FilterExpression': 'contains(id, :p) AND #s = :c',
        'ExpressionAttributeNames': {'#s': 'status'},
        'ExpressionAttributeValues': {':p': 'vkg-spider2_', ':c': 'completed'},
    }
    while True:
        page = _meta_table.scan(**scan_kw)
        for it in page.get('Items', []):
            lid = it['id']                       # vkg-spider2_<db>
            if lid.startswith('vkg-'):
                done.add(lid[len('vkg-'):])      # -> spider2_<db> (== row.db_id)
        if 'LastEvaluatedKey' not in page:
            break
        scan_kw['ExclusiveStartKey'] = page['LastEvaluatedKey']
    return done


_built_db_ids = _completed_spider2_vkg_layers()
print(f"✓ Completed VKG layers for {len(_built_db_ids)} Spider 2.0 db(s): "
      f"{sorted(_built_db_ids) or '(none)'}")
if not _built_db_ids:
    raise RuntimeError(
        "No completed vkg-spider2_* layers found. Run 19_spider2_semantic_layer.ipynb "
        "first (it builds one VKG layer per Spider 2.0 db)."
    )

# Keep rows that are (a) ingested, (b) have a built VKG layer, and (c) carry gold
# SQL (execution-scoreable). Advisory/meta rows with no gold SQL cannot be scored.
rows = [r for r in all_rows
        if r.db_id in _ingested_ids and r.db_id in _built_db_ids and r.gold_sql]

MAX_ROWS = int(os.environ.get('MAX_ROWS', '0'))
if MAX_ROWS > 0:
    rows = rows[:MAX_ROWS]
    print(f"  ⚠ MAX_ROWS={MAX_ROWS} — evaluating a {len(rows)}-row slice (0 = all)")

if not rows:
    raise RuntimeError(
        "No scoreable Spider 2.0 rows (need an ingested + VKG-built db AND gold SQL)."
    )
print(f"✓ {len(rows)} scoreable Spider 2.0 row(s) across "
      f"{len({r.db_id for r in rows})} db(s)")
pd.set_option('display.max_colwidth', None)
display(pd.DataFrame([{'db_id': r.db_id, 'difficulty': r.difficulty,
                       'question': r.question} for r in rows]).head(20))


## 3. Resolve the ontology_query runtime + the codeBased evaluator Lambda

In [ ]:
# ── Look up the ontology_query runtime ARN + the BenchmarkEvalHandler Lambda ──
# BOTH come from the {PROJECT_NAME}-agentcore stack outputs (try the -dev suffix
# too). The VKG query agent's output key is `QueryRuntimeArn` (NOT
# `OntologyQueryRuntimeArn`); the codeBased exec-metrics Lambda is
# `BenchmarkEvalHandlerArn` (export `<project>-benchmark-eval-handler-arn`).
cfn = session.client('cloudformation', region_name=region)
outputs = None
for _stack in (f'{PROJECT_NAME}-agentcore', f'{PROJECT_NAME}-dev-agentcore'):
    try:
        outputs = cfn.describe_stacks(StackName=_stack)['Stacks'][0]['Outputs']
        stack_name = _stack
        break
    except Exception:  # noqa: BLE001 — try the -dev fallback stack name
        continue
if outputs is None:
    raise RuntimeError(f"AgentCore stack not found for project {PROJECT_NAME}")
output_map = {o['OutputKey']: o['OutputValue'] for o in outputs}

ONTOLOGY_QUERY_RUNTIME_ARN = output_map['QueryRuntimeArn']
# AgentCore's batch path does NOT pass evaluatorName to the codeBased Lambda,
# so each metric is a SEPARATE function with EVAL_METRIC baked into its env.
# Resolve one ARN per metric (output keys BenchmarkEvalHandler<Metric>Arn).
BENCHMARK_EVAL_LAMBDA_ARNS = {
    'ExecutionAccuracy': output_map['BenchmarkEvalHandlerExecutionAccuracyArn'],
    'RewardVES': output_map['BenchmarkEvalHandlerRewardVESArn'],
    'SoftF1': output_map['BenchmarkEvalHandlerSoftF1Arn'],
}
# arn:aws:bedrock-agentcore:region:account:runtime/<agent_id>
agent_id = ONTOLOGY_QUERY_RUNTIME_ARN.rsplit('/', 1)[-1]

print(f"✓ From stack '{stack_name}':")
print(f"  Query Runtime ARN     : {_mask_acct(ONTOLOGY_QUERY_RUNTIME_ARN)}")
print(f"  Agent ID              : {agent_id}")
for _m, _arn in BENCHMARK_EVAL_LAMBDA_ARNS.items():
    print(f"  BenchmarkEval {_m:<17}: {_mask_acct(_arn)}")


## 4. Create the evaluators — LLM judges (SESSION) + codeBased exec metrics (TRACE)

Two evaluator families, both scored server-side:

| Evaluator | Kind | Level | Source |
|:--|:--|:--|:--|
| `GoalSuccess` | LLM-as-Judge | SESSION | `eval_judges` (VKG) |
| `FinalAnswerFaithfulness` | LLM-as-Judge | SESSION | `eval_judges` (VKG) |
| `SqlGrounded` | LLM-as-Judge | SESSION | `eval_judges` (VKG) |
| `spider2_ExecutionAccuracy` | codeBased | TRACE | `BenchmarkEvalHandler` Lambda |
| `spider2_RewardVES` | codeBased | TRACE | `BenchmarkEvalHandler` Lambda |
| `spider2_SoftF1` | codeBased | TRACE | `BenchmarkEvalHandler` Lambda |

> The codeBased evaluators branch on the evaluator NAME (the Lambda matches the
> canonical substrings `ExecutionAccuracy` / `RewardVES` / `SoftF1`), so a
> `spider2_` prefix is fine. All three point at the SAME Lambda ARN.

> **`Builtin.GoalSuccessRate` is intentionally omitted** — nb5 documents that the
> AWS-managed builtin mis-scores this deterministic-graph agent (it mistakes an
> intermediate intent/SPARQL span for the answer). The custom `GoalSuccess`
> SESSION judge replaces it. (If you want the builtin as a cross-check, append
> `'Builtin.GoalSuccessRate'` to `ALL_EVALUATOR_IDS` in the next cell — read it
> skeptically.)


In [ ]:
# ── LLM-as-Judge (SESSION) evaluators — VKG family, from eval_judges ─────────
try:
    from agents.shared.eval_judges import create_custom_judges, JUDGE_MODEL_ID
except ImportError:
    import sys; sys.path.insert(0, '..')
    from agents.shared.eval_judges import create_custom_judges, JUDGE_MODEL_ID

_cp = session.client('bedrock-agentcore-control', region_name=region)

# family='vkg' → [GoalSuccess, FinalAnswerFaithfulness, SqlGrounded] (byte-identical
# to nb5's judges). GoalSuccess is the custom replacement for the un-editable
# Builtin.GoalSuccessRate on this deterministic-graph agent.
GOAL_SUCCESS_ID, ANSWER_FAITHFUL_ID, SQL_GROUNDED_ID = create_custom_judges(
    control_client=_cp, family='vkg',
)
CUSTOM_JUDGE_IDS = [GOAL_SUCCESS_ID, ANSWER_FAITHFUL_ID, SQL_GROUNDED_ID]
print("✓ Custom SESSION LLM-as-Judge evaluators created:")
print(f"  GoalSuccess             : {GOAL_SUCCESS_ID}")
print(f"  FinalAnswerFaithfulness : {ANSWER_FAITHFUL_ID}")
print(f"  SqlGrounded             : {SQL_GROUNDED_ID}")


# ── codeBased exec-metric (TRACE) evaluators — one per metric, same Lambda ───
# The AgentCore CreateEvaluator codeBased config is
#   {'codeBased': {'lambdaConfig': {'lambdaArn': <arn>, 'lambdaTimeoutInSeconds': N}}}
# level=TRACE (one score per trace). We register three, whose names carry the
# canonical metric substrings the Lambda branches on (ExecutionAccuracy / RewardVES
# / SoftF1). A fresh suffix keeps redeploys unique.
import uuid as _uuid
_SUFFIX = _uuid.uuid4().hex[:8]
# Lambda run_sql polls up to 120s PER query and runs gold THEN predicted, so give
# the codeBased invocation headroom (max allowed is 300s).
_LAMBDA_TIMEOUT_S = 280


def _create_codebased_evaluator(*, metric_name: str) -> str:
    """Create one codeBased exec-metric evaluator (TRACE) and return its id.

    Parameters:
        metric_name: The metric substring the Lambda branches on — one of
            'ExecutionAccuracy', 'RewardVES', 'SoftF1'.
    Returns:
        The created evaluator's evaluatorId.
    """
    resp = _cp.create_evaluator(
        evaluatorName=f'spider2_{metric_name}_{_SUFFIX}',
        level='TRACE',
        evaluatorConfig={
            'codeBased': {
                'lambdaConfig': {
                    # Per-metric function ARN — EVAL_METRIC in its env
                    # tells the shared handler which metric to score.
                    'lambdaArn': BENCHMARK_EVAL_LAMBDA_ARNS[metric_name],
                    'lambdaTimeoutInSeconds': _LAMBDA_TIMEOUT_S,
                }
            }
        },
    )
    return resp['evaluatorId']


EX_ID = _create_codebased_evaluator(metric_name='ExecutionAccuracy')
RVES_ID = _create_codebased_evaluator(metric_name='RewardVES')
SOFTF1_ID = _create_codebased_evaluator(metric_name='SoftF1')
CODEBASED_IDS = [EX_ID, RVES_ID, SOFTF1_ID]
print("\n✓ codeBased exec-metric evaluators created (TRACE, same Lambda):")
print(f"  spider2_ExecutionAccuracy : {EX_ID}")
print(f"  spider2_RewardVES         : {RVES_ID}")
print(f"  spider2_SoftF1            : {SOFTF1_ID}")

# All evaluator ids the batch job scores. LLM judges (SESSION) + exec metrics
# (TRACE) — the batch API applies each at its own level from one flat list.
ALL_EVALUATOR_IDS = [*CUSTOM_JUDGE_IDS, *CODEBASED_IDS]

# Display-name map for the results tables (event names drop the trailing -<hash>).
_EVAL_NAME = {
    GOAL_SUCCESS_ID: 'GoalSuccess', ANSWER_FAITHFUL_ID: 'FinalAnswerFaithfulness',
    SQL_GROUNDED_ID: 'SqlGrounded', EX_ID: 'ExecutionAccuracy',
    RVES_ID: 'RewardVES', SOFTF1_ID: 'SoftF1',
}
_EVAL_NAME.update({_eid.rsplit('-', 1)[0]: _disp for _eid, _disp in list(_EVAL_NAME.items())})


## 5. Build the Dataset (one scenario per Spider 2.0 row) + agent_invoker

Each Spider 2.0 row is one `PredefinedScenario` (its own session), bound to its DB's
VKG layer via `ontologyId = f"vkg-{row.db_id}"`.

**Gold-packing:** each row's `expected_response` is set to
`json.dumps({"gold_sql", "db_id", "catalog"})` (via `parse_multiturn_row` +
`build_scenarios`). The batch API delivers that as the TRACE
`expectedResponse.text` — which the codeBased Lambda reads for EX / R-VES /
Soft-F1. (It also becomes the LLM judge's `FINAL_ANSWER_ASSERTION_PREFIX`
reference; see the intro for why `FinalAnswerFaithfulness` is weak on Spider 2.0.)


In [ ]:
# Per-row chat-stream invoker + dataset builder (mirrors nb5, but binds ontologyId
# PER ROW and packs the codeBased gold into expected_response).
import time
from bedrock_agentcore.evaluation.runner.invoker_types import (
    AgentInvokerInput, AgentInvokerOutput)
from bedrock_agentcore.evaluation.runner.dataset_types import Dataset
try:
    from agents.shared.eval_multiturn import (
        parse_multiturn_row, build_chat_payload, build_scenarios,
        parse_chat_stream_sse, format_agent_output, run_key)
except ImportError:
    import sys; sys.path.insert(0, '..')
    from agents.shared.eval_multiturn import (
        parse_multiturn_row, build_chat_payload, build_scenarios,
        parse_chat_stream_sse, format_agent_output, run_key)

# ── Pack the codeBased gold into each row's Expected_Answer ──────────────────
# The codeBased Lambda reads expectedResponse.text = json.dumps({gold_sql, db_id,
# catalog}); build_scenarios threads a row's Expected_Answer onto the final
# Turn.expected_response (→ the TRACE expectedResponse.text). So we set
# Expected_Answer to the packed gold. db_id (= namespace spider2_<db>) is the Athena
# database the Lambda executes gold SQL in; catalog is the federated S3 Tables
# catalog (per-row from the manifest entry). One scenario per row.
_specs = []
SCENARIO_LAYER = {}        # scenario_id -> ontologyId (vkg-<db_id>)
SCENARIO_ROW = {}          # scenario_id -> the BenchmarkRow (for the results file)
for i, row in enumerate(rows):
    entry = manifest.entry(db_id=row.db_id)
    gold_payload = json.dumps({
        'gold_sql': row.gold_sql,
        'db_id': row.db_id,          # == namespace spider2_<db> == Athena database
        'catalog': entry.catalog,    # federated S3 Tables catalog id
    })
    spec = parse_multiturn_row(
        {'Natural_Language_Question': row.question, 'Expected_Answer': gold_payload},
        index=i,
    )
    layer_id = f'vkg-{row.db_id}'    # SAME id nb19 built
    SCENARIO_LAYER[spec['scenario_id']] = layer_id
    SCENARIO_ROW[spec['scenario_id']] = row
    _specs.append(spec)

# build_scenarios takes ONE ontology_id (stamped into scenario metadata), but Spider 2.0
# rows span DBs, so call it PER spec with that row's layer id and concatenate. The
# real ontologyId binding happens in agent_invoker (below) via build_chat_payload.
scenarios = []
for spec in _specs:
    scenarios.extend(build_scenarios(
        [spec], ontology_id=SCENARIO_LAYER[spec['scenario_id']],
        simulated_enabled=False))
dataset = Dataset(scenarios=scenarios)
print(f"✓ Dataset: {len(dataset.scenarios)} Spider 2.0 scenario(s) (one per row)")


AGENT_RUNS = {}            # keyed by run_key(session_id, turn_idx)


def _scenario_id_for_session(sess: str) -> str:
    """Map a framework session id (scenario_id + '-' + uuid) back to its scenario_id."""
    for s in _specs:
        if sess and sess.startswith(s['scenario_id'] + '-'):
            return s['scenario_id']
    return sess or ''


def agent_invoker(invoker_input: AgentInvokerInput) -> AgentInvokerOutput:
    """Invoke the VKG query agent for one Spider 2.0 row via the chat-stream path.

    Binds ontologyId to the row's per-DB VKG layer (vkg-<db_id>), parses the SSE
    body, and records the predicted SQL + cost/latency into AGENT_RUNS.
    """
    sid = invoker_input.session_id
    scenario_id = _scenario_id_for_session(sid)
    ontology_id = SCENARIO_LAYER.get(scenario_id, '')
    turn_idx = sum(1 for k in AGENT_RUNS if k.startswith(f"{sid}#turn"))
    message = (invoker_input.payload if isinstance(invoker_input.payload, str)
               else json.dumps(invoker_input.payload))
    payload = build_chat_payload(message=message, session_id=sid,
                                 ontology_id=ontology_id, turn_idx=turn_idx,
                                 user_id="spider2-eval", source="eval")
    payload["mode"] = "vkg"                    # VKG path (default is semantic-rag)
    start = time.time(); err = None; parsed = {}
    try:
        raw = _invoke_runtime(ONTOLOGY_QUERY_RUNTIME_ARN, sid,
                              json.dumps(payload).encode("utf-8"))
        parsed = parse_chat_stream_sse(raw.decode("utf-8", errors="replace"))
        err = parsed.get("error")
    except Exception as exc:  # noqa: BLE001 — one row failure must not abort the batch
        err = str(exc); print(f"  ⚠ [{sid} t{turn_idx}] {err}")
    AGENT_RUNS[run_key(sid, turn_idx)] = {
        "scenario_session": sid, "scenario_id": scenario_id, "turn_idx": turn_idx,
        "ontology_id": ontology_id, "message": message,
        "answer": parsed.get("answer", ""), "agent_sql": parsed.get("sql", ""),
        "executed_sql": parsed.get("executed_sql", ""), "degraded": parsed.get("degraded"),
        "rows": parsed.get("rows", []) or [], "usage": parsed.get("usage", {}),
        "runtime_ms": parsed.get("runtime_ms"),
        "wall_clock_s": round(time.time() - start, 2), "invoke_error": err,
    }
    state = "answer" if parsed.get("sql") else ("clarify" if parsed.get("clarification") else "none")
    print(f"  {'OK' if err is None else 'XX'} [{scenario_id} {ontology_id}] "
          f"{AGENT_RUNS[run_key(sid, turn_idx)]['wall_clock_s']}s . {state}")
    return AgentInvokerOutput(agent_output=format_agent_output(parsed) or (err or ""))


## 6. Run the Batch Evaluation (server-side)

`run_dataset_evaluation` invokes the agent once per scenario (via `agent_invoker`),
waits `ingestion_delay_seconds` for CloudWatch span ingestion, submits ONE
`StartBatchEvaluation` over all six evaluators (SESSION judges + TRACE exec
metrics), and polls to completion. All scoring is in-service.


In [ ]:
from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_runner import (
    BatchEvaluationRunner,
)
from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_models import (
    BatchEvaluationRunConfig, BatchEvaluatorConfig, CloudWatchDataSourceConfig,
)

SERVICE_NAME = f"{agent_id}.DEFAULT"
RUNTIME_LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{agent_id}-DEFAULT"
SPANS_LOG_GROUP = "aws/spans"
print(f"SERVICE_NAME : {SERVICE_NAME}")
print(f"LOG GROUPS   : {SPANS_LOG_GROUP}, {RUNTIME_LOG_GROUP}")
print(f"Evaluators   : {ALL_EVALUATOR_IDS}")
print(f"Scenarios    : {len(dataset.scenarios)}")

batch_data_source = CloudWatchDataSourceConfig(
    service_names=[SERVICE_NAME],
    log_group_names=[SPANS_LOG_GROUP, RUNTIME_LOG_GROUP],
    ingestion_delay_seconds=420,   # give spans (incl. the SQL-exec span) time to land
)

batch_runner = BatchEvaluationRunner(region=region)

cfg = BatchEvaluationRunConfig(
    batch_evaluation_name=f"spider2_vkg_bench_{uuid.uuid4().hex[:8]}",
    description="Spider 2.0 VKG query-agent eval: LLM judges (SESSION) + codeBased "
                "exec metrics EX/R-VES/Soft-F1 (TRACE).",
    evaluator_config=BatchEvaluatorConfig(evaluator_ids=ALL_EVALUATOR_IDS),
    data_source=batch_data_source,
    max_concurrent_scenarios=3,    # synchronous agent, up to 900s/row
    polling_timeout_seconds=5400,  # VKG rows are multi-phase + the codeBased Lambda runs 2 Athena queries/row
    polling_interval_seconds=30,
)

print(f"\n=== {cfg.batch_evaluation_name} ===")
print("Invoking the agent per row, then evaluating server-side...")
batch_result = batch_runner.run_dataset_evaluation(
    config=cfg, dataset=dataset, agent_invoker=agent_invoker)
print(f"\n✓ Batch status: {batch_result.status}")
if getattr(batch_result, 'agent_invocation_failures', None):
    print(f"⚠ Agent invocation failures: {len(batch_result.agent_invocation_failures)}")
if batch_result.error_details:
    print(f"⚠ Error details: {batch_result.error_details}")


## 7. Results — LLM-judge aggregates + exec-metric (EX / R-VES / Soft-F1) aggregates

1. **Aggregate per-evaluator scores** — from `batch_result.evaluation_results`.
   The three codeBased evaluators' `average_score` **are** the mean EX / R-VES /
   Soft-F1 over the scored rows; the three judges' are the mean pass rates.
2. **Per-row evaluator scores** — `fetch_evaluation_events()`, joined by session id.
3. **Per-row agent detail** — predicted SQL / cost / latency from `AGENT_RUNS`.


In [ ]:
os.makedirs('../data/eval/results', exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

print(f"Batch ID : {batch_result.batch_evaluation_id}")
print(f"Status   : {batch_result.status}")

# ── 1. Aggregate per-evaluator scores (LLM judges + exec metrics) ────────────
agg_rows = []
_agg_by_name = {}
ev = batch_result.evaluation_results
if ev is not None:
    print(f"\nSessions: completed={ev.number_of_sessions_completed} "
          f"failed={ev.number_of_sessions_failed} total={ev.total_number_of_sessions}")
    for es in (ev.evaluator_summaries or []):
        stats = es.statistics
        avg = (stats.average_score if stats and stats.average_score is not None else None)
        name = _EVAL_NAME.get(es.evaluator_id, es.evaluator_id)
        _agg_by_name[name] = round(avg, 4) if avg is not None else None
        agg_rows.append({
            'Evaluator': name,
            'AvgScore': f"{avg:.3f}" if avg is not None else None,
            'Evaluated': es.total_evaluated or 0, 'Failed': es.total_failed or 0,
        })
else:
    print("\n⚠ No aggregate evaluation_results — check job status / spans.")
df_agg = pd.DataFrame(agg_rows)

# ── 2. Per-row evaluator events ──────────────────────────────────────────────
def _event_field(e: dict, key: str):
    """Read an event field from the top level or the nested 'attributes' map."""
    if key in e:
        return e[key]
    return (e.get('attributes', {}) or {}).get(key)

events = []
for _attempt in range(6):
    try:
        events = batch_runner.fetch_evaluation_events(batch_result)
        if events:
            break
    except (LookupError, ValueError) as exc:
        print(f"  per-row events not ready yet ({exc}); retrying in 20s...")
    time.sleep(20)

event_rows = []
for e in events:
    sess = _event_field(e, 'session.id') or _event_field(e, 'gen_ai.session.id')
    name = _event_field(e, 'gen_ai.evaluation.name')
    sid = _scenario_id_for_session(sess)
    _row = SCENARIO_ROW.get(sid)
    event_rows.append({
        'scenario_id': sid,
        'db_id': getattr(_row, 'db_id', ''),
        'difficulty': getattr(_row, 'difficulty', '') or '',
        'question': getattr(_row, 'question', ''),
        'evaluator': _EVAL_NAME.get(name, name),
        'score': _event_field(e, 'gen_ai.evaluation.score.value'),
        'label': _event_field(e, 'gen_ai.evaluation.score.label'),
        'explanation': str(_event_field(e, 'gen_ai.evaluation.explanation') or ''),
    })
df_events = pd.DataFrame(event_rows)
print(f"\nPer-row evaluator events: {len(df_events)}")

# ── Per-difficulty EX/R-VES/Soft-F1 from the exec-metric events ──────────────
_EXEC_NAMES = {'ExecutionAccuracy', 'RewardVES', 'SoftF1'}
by_diff = {}
for r in event_rows:
    if r['evaluator'] not in _EXEC_NAMES or not isinstance(r['score'], (int, float)):
        continue
    by_diff.setdefault(r['difficulty'] or 'unknown', {}).setdefault(
        r['evaluator'], []).append(r['score'])
exec_by_difficulty = {
    band: {m: round(sum(v) / len(v), 4) for m, v in metrics.items()}
    for band, metrics in by_diff.items()
}

# ── 3. Per-row agent detail (predicted SQL / cost / latency) ─────────────────
agent_detail = []
for rec in AGENT_RUNS.values():
    agent_detail.append({
        'scenario_id': rec['scenario_id'], 'ontology_id': rec['ontology_id'],
        'question': rec['message'], 'agent_sql': rec['agent_sql'],
        'executed_sql': rec['executed_sql'], 'degraded': rec['degraded'],
        'wall_clock_s': rec['wall_clock_s'], 'error': rec['invoke_error'],
    })
df_agent = pd.DataFrame(agent_detail)

# ── Persist ──────────────────────────────────────────────────────────────────
combined_file = f"../data/eval/results/spider2_vkg_benchmark_eval_{timestamp}.json"
combined = {
    'benchmark': 'spider2',
    'agent': 'ontology_query_agent',
    'agent_id': agent_id,
    'runtime_arn': ONTOLOGY_QUERY_RUNTIME_ARN,
    'benchmark_eval_lambda_arns': BENCHMARK_EVAL_LAMBDA_ARNS,
    'batch_evaluation_id': batch_result.batch_evaluation_id,
    'batch_evaluation_arn': batch_result.batch_evaluation_arn,
    'status': str(batch_result.status),
    'evaluator_ids': ALL_EVALUATOR_IDS,
    'llm_judges': {'GoalSuccess': GOAL_SUCCESS_ID,
                   'FinalAnswerFaithfulness': ANSWER_FAITHFUL_ID,
                   'SqlGrounded': SQL_GROUNDED_ID},
    'exec_metric_evaluators': {'ExecutionAccuracy': EX_ID, 'RewardVES': RVES_ID,
                               'SoftF1': SOFTF1_ID},
    # Headline: the LLM-judge aggregates AND the exec-metric (EX/R-VES/Soft-F1) aggregates.
    'llm_judge_aggregates': {k: _agg_by_name.get(k) for k in
                             ('GoalSuccess', 'FinalAnswerFaithfulness', 'SqlGrounded')},
    'exec_metric_aggregates': {k: _agg_by_name.get(k) for k in
                               ('ExecutionAccuracy', 'RewardVES', 'SoftF1')},
    'exec_metrics_by_difficulty': exec_by_difficulty,
    'aggregate_summaries': agg_rows,
    'per_row_events': event_rows,
    'agent_detail': agent_detail,
    'n_rows': len(rows),
}
combined = _redact_account_ids(combined)
with open(combined_file, 'w') as f:
    json.dump(combined, f, indent=2, default=str)
print(f"\n✓ Wrote {combined_file}")

print("\n=== Aggregate per-evaluator scores ===")
display(df_agg)
_ex = combined['exec_metric_aggregates']
print(f"\nHeadline exec metrics (n={len(rows)}): "
      f"EX={_ex.get('ExecutionAccuracy')}  R-VES={_ex.get('RewardVES')}  "
      f"Soft-F1={_ex.get('SoftF1')}")
if exec_by_difficulty:
    print("\nExec metrics by difficulty:")
    for band in sorted(exec_by_difficulty):
        m = exec_by_difficulty[band]
        print(f"  {band:>12}: EX={m.get('ExecutionAccuracy')}  "
              f"R-VES={m.get('RewardVES')}  Soft-F1={m.get('SoftF1')}")
print("\n=== Per-row evaluator scores ===")
display(df_events)
print("=== Per-row agent detail ===")
display(df_agent)


## Summary

This notebook runs the **Spider 2.0-lite (SQLite subset)** benchmark through the
deployed **ontology_query** (VKG) agent and scores each answer **server-side, two
ways**:

1. **LLM judges (SESSION)** — `GoalSuccess`, `FinalAnswerFaithfulness`,
   `SqlGrounded` from `eval_judges` (VKG family), identical to nb5.
2. **codeBased exec metrics (TRACE)** — `spider2_ExecutionAccuracy` (EX),
   `spider2_RewardVES` (R-VES), `spider2_SoftF1`, three `codeBased` evaluators
   pointing at the `BenchmarkEvalHandler` Lambda (`BenchmarkEvalHandlerArn`), which
   extracts the predicted SQL from the trace, reads the packed gold from
   `expectedResponse.text`, and executes both on Athena.

### Gold-packing / dual-scoring trade-off
Spider 2.0's gold is **SQL** (transpiled SQLite→Trino), not prose — like BIRD. The
gold JSON `{gold_sql, db_id, catalog}` is packed into each scenario's
`expected_response` (→ TRACE `expectedResponse.text`), which is what the codeBased
Lambda reads — so **EX / R-VES / Soft-F1 are the correct, primary Spider 2.0
signal**. Because there is no prose gold answer, that same packed JSON is what the
LLM `FinalAnswerFaithfulness` judge sees as its expected answer → **weak for
Spider 2.0** (accepted). `GoalSuccess` and `SqlGrounded` don't depend on a gold NL
answer and remain meaningful.

### Layer binding
`ontologyId = f"vkg-{row.db_id}"` — each Spider 2.0 row's `db_id` is its namespace
`spider2_<db>`, so this resolves to `vkg-spider2_<db>`, the exact id
`19_spider2_semantic_layer.ipynb` built. Rows are filtered to DBs with a completed
`vkg-spider2_*` layer.

### Notes
- **R-VES caveat**: the Lambda times both gold and predicted SQL on Athena;
  there is no external reference timing for Spider 2.0, so read R-VES as a
  within-arm relative signal. **EX** and **Soft-F1** are the portable metrics.
- Results → `../data/eval/results/spider2_vkg_benchmark_eval_*.json` (LLM-judge
  aggregates + exec-metric aggregates + per-difficulty EX/R-VES/Soft-F1 +
  per-row detail).
